# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codingsheep17/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#initializing the repo
import os

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/codingsheep17/flyrank-ml-internship.git

os.chdir("flyrank-ml-internship")
print(os.getcwd())

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 229 (delta 112), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (229/229), 2.50 MiB | 16.21 MiB/s, done.
Resolving deltas: 100% (112/112), done.
/content/flyrank-ml-internship


In [2]:
#installing the dataset
!pip install duckdb huggingface_hub -q

from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded:", "HF_TOKEN" in os.environ)

Token loaded: True


In [3]:
import duckdb

con = duckdb.connect()
con.sql(f"""
CREATE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")
print("DuckDB secret configured")

DuckDB secret configured


In [5]:
dim_content_schema = con.sql("""
DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' LIMIT 1
""").df()

print(dim_content_schema.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [6]:
staleness_check = con.sql("""
SELECT
    CASE
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 90 THEN 'fresh'
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 180 THEN 'aging'
        WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 365 THEN 'stale'
        ELSE 'very_stale'
    END as staleness_bucket,
    COUNT(*) as n,
    AVG(f.gsc_clicks) as avg_clicks
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d
ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE
GROUP BY staleness_bucket
ORDER BY avg_clicks DESC
""").df()

staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_clicks
0,fresh,3599822,0.227930
1,aging,9622,0.135003
2,stale,1617,0.016698


In [7]:
#now the ctr
ctr_position_check = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN 'top_3'
        WHEN gsc_avg_position <= 10 THEN 'page_1'
        WHEN gsc_avg_position <= 20 THEN 'page_2'
        ELSE 'deep'
    END as position_bucket,
    COUNT(*) as n,
    AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY position_bucket
ORDER BY avg_ctr DESC
""").df()

ctr_position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,top_3,727362,0.004756
1,page_1,1456122,0.003473
2,page_2,519223,0.002770
3,deep,908354,0.001289


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: Staleness — Verdict: CONFIRMED
Average clicks clearly decline as content gets staler: fresh pages average 0.228 clicks, aging pages 0.135, stale pages just 0.017 — roughly a 13x drop from fresh to stale. This confirms the signal behind FlyRank's refresh flags: staleness is genuinely associated with declining performance in this data. Note: no pages fell into "very_stale" (365+ days) in this slice — worth noting as a limitation of this month's sample.

Signal 2: CTR vs Position — Verdict: CONFIRMED
Average CTR clearly declines with worse position: top_3 pages average 0.48% CTR, dropping to 0.35% (page_1), 0.28% (page_2), and just 0.13% for deep results — roughly a 3.7x drop from top_3 to deep. This confirms the signal behind FlyRank's CTR-fix logic: a page's CTR is strongly tied to its ranking position, so any "low CTR" flag must account for position tier rather than comparing raw CTR across all pages equally.

Actual Rule

Baseline rule: Flag pages that are both stale AND underperforming CTR for their position tier — since both signals were independently confirmed above, combining them targets pages with the strongest evidence of needing review.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_df = con.sql("""
SELECT
    f.content_hash_id,
    f.client_hash_id,
    d.content_updated_date,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    (f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) as ctr,
    DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') as days_stale
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d
ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE AND f.gsc_impressions >= 50
""").df()

print(f"Rows after impressions filter: {len(baseline_df)}")
baseline_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after impressions filter: 1037442


,content_hash_id,client_hash_id,content_updated_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,days_stale
0,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,2026-06-29,321,0,6.246106,0.000,-90
1,content_26f5092ee7f70d45,client_62f4a7e64f5e0096,2026-06-29,87,0,6.321839,0.000,-90
2,content_9e7c70abfbae371e,client_62f4a7e64f5e0096,2026-06-29,139,0,5.856115,0.000,-90
3,content_b4de71c8ef5c4791,client_62f4a7e64f5e0096,2026-06-29,125,2,3.224000,0.016,-90
4,content_85b1be9944e4e19d,client_62f4a7e64f5e0096,2026-06-29,50,0,7.120000,0.000,-90


In [9]:
baseline_df = con.sql("""
SELECT
    f.content_hash_id,
    f.client_hash_id,
    d.content_updated_date,
    f.report_date,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    (f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0)) as ctr,
    DATE_DIFF('day', d.content_updated_date, f.report_date) as days_stale
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d
ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE
  AND f.gsc_impressions >= 50
  AND d.content_updated_date <= f.report_date
""").df()

print(f"Rows after fixes: {len(baseline_df)}")
baseline_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after fixes: 145678


,content_hash_id,client_hash_id,content_updated_date,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,days_stale
0,content_68c865586c701826,client_73cda7b4e4f265ea,2026-02-25,2026-03-01,298,0,2.550336,0.0,4
1,content_1cf759319adacd7e,client_73cda7b4e4f265ea,2026-02-25,2026-03-01,141,0,2.716312,0.0,4
2,content_f75ac8fc44bc96cd,client_73cda7b4e4f265ea,2026-02-25,2026-03-01,73,0,1.863014,0.0,4
3,content_35d1f03821390fb9,client_73cda7b4e4f265ea,2026-02-25,2026-03-01,50,0,1.660000,0.0,4
4,content_3ff6ae6ce85bc2e6,client_73cda7b4e4f265ea,2026-02-25,2026-03-01,71,0,5.070423,0.0,4


In [10]:
import numpy as np

# Position bucket for CTR context (needed to compare CTR fairly, per Signal 2 finding)
def position_bucket(pos):
    if pos <= 3: return 'top_3'
    elif pos <= 10: return 'page_1'
    elif pos <= 20: return 'page_2'
    else: return 'deep'

baseline_df['position_bucket'] = baseline_df['gsc_avg_position'].apply(position_bucket)

# Expected CTR per bucket (from Signal 2 results)
expected_ctr = {'top_3': 0.004756, 'page_1': 0.003473, 'page_2': 0.002770, 'deep': 0.001289}
baseline_df['expected_ctr'] = baseline_df['position_bucket'].map(expected_ctr)
baseline_df['ctr_gap'] = baseline_df['expected_ctr'] - baseline_df['ctr']

# Normalize staleness and ctr_gap to 0-1, combine into one score
baseline_df['staleness_norm'] = (baseline_df['days_stale'] - baseline_df['days_stale'].min()) / (baseline_df['days_stale'].max() - baseline_df['days_stale'].min())
baseline_df['ctr_gap_norm'] = (baseline_df['ctr_gap'] - baseline_df['ctr_gap'].min()) / (baseline_df['ctr_gap'].max() - baseline_df['ctr_gap'].min())

baseline_df['score'] = 0.5 * baseline_df['staleness_norm'] + 0.5 * baseline_df['ctr_gap_norm']

# Reason code
def reason_code(row):
    if row['days_stale'] > 180 and row['ctr_gap'] > 0:
        return 'stale_and_underperforming_ctr'
    elif row['days_stale'] > 180:
        return 'stale_visible_page'
    elif row['ctr_gap'] > 0:
        return 'low_ctr_for_position'
    else:
        return 'low_priority'

baseline_df['reason_code'] = baseline_df.apply(reason_code, axis=1)
baseline_df['action'] = 'review_for_refresh'

baseline_df = baseline_df.sort_values('score', ascending=False)
baseline_df.head(10)

,content_hash_id,client_hash_id,content_updated_date,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,days_stale,position_bucket,expected_ctr,ctr_gap,staleness_norm,ctr_gap_norm,score,reason_code,action
65013,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-31,135,0,4.962963,0.0,264,page_1,0.003473,0.003473,1.000000,0.988656,0.994328,stale_and_underperforming_ctr,review_for_refresh
144929,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-30,103,0,4.203883,0.0,263,page_1,0.003473,0.003473,0.996212,0.988656,0.992434,stale_and_underperforming_ctr,review_for_refresh
144099,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-27,200,0,2.640000,0.0,260,top_3,0.004756,0.004756,0.984848,1.000000,0.992424,stale_and_underperforming_ctr,review_for_refresh
64834,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-29,115,0,4.400000,0.0,262,page_1,0.003473,0.003473,0.992424,0.988656,0.990540,stale_and_underperforming_ctr,review_for_refresh
64631,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-28,348,0,3.324713,0.0,261,page_1,0.003473,0.003473,0.988636,0.988656,0.988646,stale_and_underperforming_ctr,review_for_refresh
145278,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-26,67,0,3.567164,0.0,259,page_1,0.003473,0.003473,0.981061,0.988656,0.984858,stale_and_underperforming_ctr,review_for_refresh
144350,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-25,102,0,3.960784,0.0,258,page_1,0.003473,0.003473,0.977273,0.988656,0.982964,stale_and_underperforming_ctr,review_for_refresh
136836,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-24,328,0,3.579268,0.0,257,page_1,0.003473,0.003473,0.973485,0.988656,0.981070,stale_and_underperforming_ctr,review_for_refresh
63127,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-23,76,0,5.092105,0.0,256,page_1,0.003473,0.003473,0.969697,0.988656,0.979176,stale_and_underperforming_ctr,review_for_refresh
136388,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-22,81,0,4.432099,0.0,255,page_1,0.003473,0.003473,0.965909,0.988656,0.977282,stale_and_underperforming_ctr,review_for_refresh


In [11]:
# Keep only the most recent day per page, so the queue is one row per page
baseline_df_dedup = baseline_df.sort_values('report_date').groupby('content_hash_id').tail(1)
baseline_df_dedup = baseline_df_dedup.sort_values('score', ascending=False)

print(f"Rows before dedup: {len(baseline_df)}, after dedup: {len(baseline_df_dedup)}")
baseline_df_dedup.head(10)

Rows before dedup: 145678, after dedup: 9895


,content_hash_id,client_hash_id,content_updated_date,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,days_stale,position_bucket,expected_ctr,ctr_gap,staleness_norm,ctr_gap_norm,score,reason_code,action
65013,content_42ce26be1ec6be00,client_c182d11e4862a37d,2025-07-10,2026-03-31,135,0,4.962963,0.000000,264,page_1,0.003473,0.003473,1.000000,0.988656,0.994328,stale_and_underperforming_ctr,review_for_refresh
144919,content_5e8b20b3e231a7db,client_65de48885f4ef01b,2025-08-09,2026-03-30,69,0,6.492754,0.000000,233,page_1,0.003473,0.003473,0.882576,0.988656,0.935616,stale_and_underperforming_ctr,review_for_refresh
144920,content_6ab29e527f71c978,client_65de48885f4ef01b,2025-08-09,2026-03-30,90,0,7.377778,0.000000,233,page_1,0.003473,0.003473,0.882576,0.988656,0.935616,stale_and_underperforming_ctr,review_for_refresh
65014,content_bea86ce3455100b0,client_c182d11e4862a37d,2025-08-11,2026-03-31,61,0,6.901639,0.000000,232,page_1,0.003473,0.003473,0.878788,0.988656,0.933722,stale_and_underperforming_ctr,review_for_refresh
145104,content_eba53d72e18a9f93,client_65de48885f4ef01b,2025-08-12,2026-03-28,50,0,5.860000,0.000000,228,page_1,0.003473,0.003473,0.863636,0.988656,0.926146,stale_and_underperforming_ctr,review_for_refresh
83687,content_5120dcbbb086843d,client_c182d11e4862a37d,2025-07-27,2026-03-05,327,0,5.220183,0.000000,221,page_1,0.003473,0.003473,0.837121,0.988656,0.912888,stale_and_underperforming_ctr,review_for_refresh
18726,content_7d986479e843e4cd,client_c182d11e4862a37d,2025-07-28,2026-03-03,52,0,4.057692,0.000000,218,page_1,0.003473,0.003473,0.825758,0.988656,0.907207,stale_and_underperforming_ctr,review_for_refresh
145272,content_fb428c6e1ca78da4,client_65de48885f4ef01b,2025-08-12,2026-03-26,55,1,3.490909,0.018182,226,page_1,0.003473,-0.014709,0.856061,0.827893,0.841977,stale_visible_page,review_for_refresh
65039,content_19daa2f24df1882d,client_157ffe4d4a595515,2025-10-06,2026-03-31,293,0,8.935154,0.000000,176,page_1,0.003473,0.003473,0.666667,0.988656,0.827661,low_ctr_for_position,review_for_refresh
145279,content_e2b50fec4f99bda5,client_3ffa76342f366962,2025-11-10,2026-03-31,109,0,7.733945,0.000000,141,page_1,0.003473,0.003473,0.534091,0.988656,0.761373,low_ctr_for_position,review_for_refresh


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("work/outputs", exist_ok=True)

output_cols = ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action']
baseline_df_dedup[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Saved {len(baseline_df_dedup)} rows to work/outputs/baseline_action_score.csv")

Saved 9895 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.